# Analisis Whisper turbo vs small en video de Azzaro

Objetivo: comparar `turbo` contra `small` en un video puntual, contar las diferencias de palabras y revisar los pedazos de video donde no coinciden.

El notebook no decide automaticamente quien tiene razon. Genera una tabla y clips cortos para que hagamos revision manual.

## 1. Configuracion

- Si queres bajar el video desde YouTube, completa `VIDEO_URL`.
- Si ya tenes el video local, completa `VIDEO_PATH` y deja `VIDEO_URL = None`.
- Las transcripciones se cachean en `evaluation/outputs/azzaro_whisper/transcripts/`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Video, display

from evaluation.src.whisper_model_comparison import (
    alinear_diferencias,
    descargar_video_yt,
    exportar_clips_diferencias,
    extraer_palabras,
    resumen_diferencias,
    transcribir_whisper,
)

VIDEO_URL = "PEGAR_URL_DE_AZZARO"
VIDEO_PATH = None

# En Mac suele servir si YouTube pide login. Si no hace falta, poner None.
COOKIES_FROM_BROWSER = "chrome"

WORKDIR = ROOT / "evaluation" / "outputs" / "azzaro_whisper"
FORCE_TRANSCRIBE = False
MAX_DIFFERENCE_CLIPS = 40
MARGIN_SECONDS = 1.25

WORKDIR

## 2. Descargar o cargar video

In [ ]:
if VIDEO_PATH:
    video_path = Path(VIDEO_PATH).expanduser().resolve()
else:
    video_path = descargar_video_yt(
        VIDEO_URL,
        WORKDIR / "videos",
        nombre_base="azzaro_comparacion",
        cookies_from_browser=COOKIES_FROM_BROWSER,
    )

video_path

## 3. Transcribir con ambos modelos

Esto puede tardar. Si ya existe el JSON cacheado, se reutiliza.

In [ ]:
resultados = {}
for model_name in ("turbo", "small"):
    print(f"Transcribiendo / cargando cache: {model_name}")
    resultados[model_name] = transcribir_whisper(
        video_path,
        model_name=model_name,
        output_dir=WORKDIR / "transcripts",
        force=FORCE_TRANSCRIBE,
    )

list(resultados)

## 4. Alinear palabras y contar diferencias

In [ ]:
palabras_turbo = extraer_palabras(resultados["turbo"], "turbo")
palabras_small = extraer_palabras(resultados["small"], "small")

diferencias = alinear_diferencias(palabras_turbo, palabras_small, contexto=5)
resumen = resumen_diferencias(diferencias, palabras_turbo, palabras_small)

display(pd.DataFrame([resumen]))

In [ ]:
df = pd.DataFrame(diferencias)
columnas = [
    "diff_id",
    "tipo",
    "start",
    "end",
    "duration",
    "turbo_text",
    "small_text",
    "contexto_turbo",
    "contexto_small",
]

display(df[columnas].head(30))

csv_path = WORKDIR / "diferencias_turbo_vs_small.csv"
df.to_csv(csv_path, index=False)
csv_path

## 5. Exportar clips para revisar manualmente

Cada clip cubre la diferencia mas un margen antes y despues.

In [ ]:
diferencias_con_clips = exportar_clips_diferencias(
    video_path,
    diferencias,
    output_dir=WORKDIR / "clips_diferencias",
    margen=MARGIN_SECONDS,
    max_clips=MAX_DIFFERENCE_CLIPS,
)

df_clips = pd.DataFrame(diferencias_con_clips)
clips_csv_path = WORKDIR / "diferencias_con_clips.csv"
df_clips.to_csv(clips_csv_path, index=False)

display(df_clips[["diff_id", "start", "end", "turbo_text", "small_text", "clip_path"]].head(20))
clips_csv_path

## 6. Reproducir una diferencia

Cambiar `IDX` para mirar otra fila de `df_clips`.

In [ ]:
IDX = 0

row = df_clips.iloc[IDX]
print("diff_id:", row["diff_id"])
print("tiempo:", row["start"], "-", row["end"])
print("turbo:", row["turbo_text"])
print("small:", row["small_text"])
print("contexto turbo:", row["contexto_turbo"])
print("contexto small:", row["contexto_small"])

display(Video(row["clip_path"], embed=True, html_attributes="controls"))

## 7. Planilla de revision manual

La idea es completar `gana` con `turbo`, `small`, `ninguno` o `empate`, y dejar una nota si hace falta.

In [ ]:
revision = df_clips.copy()
revision["gana"] = ""
revision["nota"] = ""

revision_path = WORKDIR / "revision_manual_turbo_vs_small.csv"
revision.to_csv(revision_path, index=False)
revision_path